In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
from google.colab import files
import imageio
import os

In [ ]:
# Load MoveNet model from TensorFlow Hub
movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")

In [ ]:
# Keypoint labels for pose detection
keypoint_names = ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder',
                  'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
                  'left_knee', 'right_knee', 'left_ankle', 'right_ankle']

# Define connections between keypoints for visualization
connections = [(0, 1), (0, 2), (1, 3), (2, 4), (0, 5), (0, 6), (5, 7), (7, 9), (6, 8), (8, 10),
               (5, 6), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]

In [ ]:
# Upload image file in Colab
uploaded = files.upload()
# Automatically pick the first uploaded file
image_path = next(iter(uploaded))  # Image path is now available in Colab as a string

In [ ]:
# Function to run pose detection
def detect_pose_static(image_path):
    image = cv2.imread(image_path)  # Read image
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
    image_resized = tf.image.resize_with_pad(tf.expand_dims(image_rgb, axis=0), 192, 192)
    image_np = image_resized.numpy().astype(np.int32)
    outputs = movenet.signatures["serving_default"](tf.constant(image_np))
    keypoints = outputs['output_0'].numpy()
    return keypoints

In [ ]:
# Function to visualize keypoints
def visualize_pose_static(image_path, keypoints):
    image = cv2.imread(image_path)
    keypoints = np.array(keypoints)
    if keypoints.shape == (1, 1, 17, 3):
        keypoints = keypoints[0, 0]
        for kp in keypoints:
            x = int(kp[1] * image.shape[1])
            y = int(kp[0] * image.shape[0])
            cv2.circle(image, (x, y), 12, (255, 0, 0), -1)
        for connection in connections:
            start_point = (int(keypoints[connection[0], 1] * image.shape[1]),
                           int(keypoints[connection[0], 0] * image.shape[0]))
            end_point = (int(keypoints[connection[1], 1] * image.shape[1]),
                         int(keypoints[connection[1], 0] * image.shape[0]))
            cv2.line(image, start_point, end_point, (0, 0, 255), 8)
        cv2_imshow(image)
    else:
        print("Unexpected shape of keypoints array:", keypoints.shape)

In [ ]:
# --- USER: Your uploaded image will be used here ---
# If running outside Colab, replace 'image_path' with your local image path
# Example: image_path = "your_image.jpg"
static_keypoints = detect_pose_static(image_path)
visualize_pose_static(image_path, static_keypoints)